In [30]:
import pandas as pd
import matplotlib.pyplot as plt

In [31]:
df = pd.read_csv('../data/raw/media_performance_raw.csv')
print(df.groupby('channel')['conversions'].agg(['mean','median']))

                  mean  median
channel                       
Display       6.247956     6.0
Email        24.010944    22.0
Paid Search  88.791545    82.5
Paid Social  27.272069    25.0
PaidSearch   99.823529    90.0
Paid_Search  78.941176    77.0
Social       22.227273    21.0
Video         6.688797     6.0
paid search  81.941176    78.0
paid_social  30.047619    25.0


In [32]:
df['channel'].value_counts()

channel
Display        734
Email          731
Video          723
Paid Social    691
Paid Search    686
Social          22
paid_social     21
Paid_Search     17
paid search     17
PaidSearch      17
Name: count, dtype: int64

In [33]:
df[df['channel'] == 'Social']['spend'].describe()

count      22.000000
mean      820.267727
std       211.989291
min       524.340000
25%       649.605000
50%       774.630000
75%       965.250000
max      1247.460000
Name: spend, dtype: float64

### Hallazgo 01 — Etiquetas inconsistentes en `channel`

**Detección:** un `groupby('channel')` para comparar media/mediana reveló 10
etiquetas donde debían existir 5 canales.

**Diagnóstico:** `value_counts()` mostró un corte natural — 5 etiquetas con ~700
filas (canónicas) y 5 con 17-22 filas (variantes mal escritas: mayúsculas,
guiones, espacios).

**Caso ambiguo:** `Social` (22 filas) podía ser variante de `Paid Social` o un
canal orgánico legítimo. Se verificó con `spend.describe()`: min = 524 (ningún
cero) → es tráfico pagado → variante de `Paid Social`.

**Fix:** `.replace()` con diccionario de mapeo. Resultado: 5 canales canónicos.

In [34]:
mapa_canales = {
    'PaidSearch': 'Paid Search',
    'Paid_Search': 'Paid Search',
    'paid search': 'Paid Search',
    'Social': 'Paid Social',
    'paid_social': 'Paid Social'
}

df['channel'] = df['channel'].replace(mapa_canales)

In [35]:
print(df['channel'].value_counts())

channel
Paid Search    737
Paid Social    734
Display        734
Email          731
Video          723
Name: count, dtype: int64


In [36]:
# check if mean and median diverge by channel (signal of outliers)
df.groupby('channel')['conversions'].agg(['mean', 'median'])

,mean,median
channel,,
Display,6.247956,6.0
Email,24.010944,22.0
Paid Search,88.660787,82.0
Paid Social,27.200272,24.0
Video,6.688797,6.0


In [37]:
# inspect the distribution edges (min, max, quartiles) per channel
df.groupby('channel')['conversions'].describe()

,count,mean,std,min,25%,50%,75%,max
channel,,,,,,,,
Display,734.0,6.247956,2.274742,2.0,5.0,6.0,7.0,25.0
Email,731.0,24.010944,9.714973,9.0,18.0,22.0,27.5,109.0
Paid Search,737.0,88.660787,33.923051,37.0,67.0,82.0,102.0,336.0
Paid Social,734.0,27.200272,11.049286,11.0,21.0,24.0,31.0,130.0
Video,723.0,6.688797,2.611762,3.0,5.0,6.0,8.0,23.0


In [38]:
# isolate the specific row behind the extreme max value in Paid Search
df[(df['channel'] == 'Paid Search') & (df['conversions'] == 336)]

,date,channel,impressions,clicks,spend,conversions,revenue
33,2024-12-02,Paid Search,118380,4069,9615.63,336,23627.29


In [39]:
df['date'].sample(20)

907     2024-02-11
2426    2023-06-16
3636    2024-01-27
3534    2023-02-09
3291    2023-12-09
83      2024-03-23
2499    2023-10-24
3055    2024-12-21
910     2024-02-19
2640    2024-10-28
369     2024-11-07
2257    2024-03-16
3181    2024-12-03
971     2023-11-05
248     2023-07-09
1220    2024-04-23
1739    2024-02-26
1147    2024-03-14
845     2024-09-27
1992    2023-09-14
Name: date, dtype: str

In [40]:
df[df['date'].str.contains('/')]

,date,channel,impressions,clicks,spend,conversions,revenue
396,06/20/2024,Paid Social,80302,1001,628.35,24,1564.73
742,12/04/2024,Email,27668,634,37.13,22,1723.98
1115,07/08/2024,Video,46261,327,273.79,6,514.11
1389,12/21/2023,Email,29318,777,49.71,37,2934.42
1684,04/16/2024,Paid Social,84742,1091,811.79,22,1311.16
1924,01/25/2024,Video,35835,280,266.50,4,306.05
2099,11/03/2024,Display,194105,741,334.52,8,452.56
2178,05/25/2024,Video,45662,382,290.32,7,589.00
3157,01/18/2023,Video,30349,230,209.77,4,327.91
3368,01/06/2024,Email,13897,303,17.83,16,1536.17


In [41]:
# convert date column from text to real datetime dtype
df['date'] = pd.to_datetime(df['date'])

ValueError: time data "06/20/2024" doesn't match format "%Y-%m-%d". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.